In [ ]:
# this mounts your Google Drive to the Colab VM.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# enter the foldername in your Drive where you have saved the unzipped
# assignment folder, e.g. 'ece697ls/assignments/assignment3/'
FOLDERNAME = None
assert FOLDERNAME is not None, "[!] Enter the foldername."

# now that we've mounted your Drive, this ensures that
# the Python interpreter of the Colab VM can load
# python files from within it.
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

%cd /content

# Pruning

After learning, neural networks have modified and learned a set of parameters to perform our classification task. However, such parameters are costly to maintain and do not hold the same importance.

Wouldn't it be great could optimize our resource usage by dropping less important values ? This is where pruning comes into play.

Pruning is a technique that cuts off parameters/structures from a model to increase sparcity and decrease overall model size, similar to cutting leafs or branches from bushes and trees. This process can lead to smaller memory consumption with minimal accuracy reduction. Moreover, pruning the network may also provide a speedup since there will be less operations being performed.

The pruning process can be performed during the end of an epoch of training or after training is complete. Experimenting to find out which way works the best is part of the fun !

In [3]:
from ece662.pruning_helper import test_model, load_model
from ece662.data_utils import get_CINIC10_data
import os

Below we will load a pre-trained model for you to work on. If you prefer, you can save your own model from the previous Tensorflow/Pytorch task and load it here.

In [4]:
#This code may take a while to execute as it is training a network form scratch

data = get_CINIC10_data()
mode = 'torch'#torch or tensorflow

test_data = [data['X_test'],data['y_test']]

path = os.path.join(f"ece662/models/{mode}.model")

model = load_model(path,mode=mode)
test_model(model,test_data,mode=mode)

Test Acc: 0.5098


## Unstructured Pruning

Unstructured Pruning is usually related to the pruning of weights in neural networks. The general idea is to select a set of weights according to a policy and setting them up to zero. 

Common policies are random weight selection or selecting the smallers weights. 
Unstructured Pruning can be performed in one or multiple layers within the same network.

Altough in theory Unstructured Pruning should decrease the number of operations performed during execution there should be explicit support within the framework or hardware to bypass such operations, otherwise it will just operated over zero.

### Perform Pruning

Using the model trained in the previous step using pytorch, perform unstructured pruning in the weights of the model by removing x% of the smallest weights. 

*   Increment global pruning by 10% until reaching total of 80% pruned weights
*   Perform inference at the end of each pruning and observe the impact into the accuracy.


Note: The percentages are related to the entire model, not per layer.



In [ ]:
################################################################################
# TODO: Perform unstructured Pruning over the trained model using 3 different  
# prunning percentages.                                
################################################################################
# *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

import copy
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune

model.eval()
PCTS = (0.4, 0.5, 0.6, 0.8)

def conv_fc_weights(mod: nn.Module):
    # yield (module, 'weight') pairs for all Conv2d/Linear layers 
    for m in mod.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            yield (m, "weight")

def make_pruned_copy(src_model: nn.Module, total_sparsity: float) -> nn.Module:
    m = copy.deepcopy(src_model).eval()
    params = list(conv_fc_weights(m))
    prune.global_unstructured(params, pruning_method=prune.L1Unstructured, amount=total_sparsity)
    for mod, _ in params:  # make pruning permanent
        prune.remove(mod, "weight")
    return m

def run_once(pct: float):
    pruned = make_pruned_copy(model, pct)
    print(f"Pruned {int(pct*100)}% of weights:")
    test_model(pruned, test_data, mode="torch")
    print()

# execute for all targets
for p in PCTS:
    run_once(p)


# *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
################################################################################
#                              END OF YOUR CODE                                #
################################################################################

Pruned 40% of weights:
Test Acc: 0.5108

Pruned 50% of weights:
Test Acc: 0.4948

Pruned 60% of weights:
Test Acc: 0.5014

Pruned 80% of weights:
Test Acc: 0.4242



## Inline Question 1:

What happened with the accuracy as the % of pruning increased ?
Why was that the case?


## Answer: 

As the percent of pruning increases, the accuracy decreases. This is expected, since pruning removes weights from the model which naturally makes it a less accurate predictor. The reason that 80% pruned is so much less accurate that 40% prune is because most neural networks have a very large amount of redundant weights and connections which don't contribute significantly to the actual efficacy of the model and can thus be safely removed with only minimal loss of accuracy. However, an 80% prune starts removing weights of actual significance. There aren't a lot of these weights, but in an 80% unstructured prune some of these key weights will inevtiably be lost and result in noticable dips in accuracy.

## Structured Pruning

Structured Pruning consists of removing a bigger chunk of the network parameters at the same time. Instead of removing only a few weights, it is commonplace to remove entire neurons. 

For example, in Convolutional Layers, removing filters can be beneficial to improve performance as it greatly decreases the amount of computation performed. However, some of these changes may affect output dimensions which may be carried over to other parts of the network. Therefore, when performing structured pruning one must always be aware of which parameters are going to be affected.

Using the previously trained model in the CINIC-10, perform Structured Prunning only in the Convolution layers of the DNN.

In [ ]:
################################################################################
# TODO: Perform unstrucuted Pruning over the trained model using 3 different  
# prunning percentages.                                
################################################################################
# *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

def conv_layers_only(net: nn.Module):
    """Yield Conv2d modules for structured (filter) pruning."""
    for m in net.modules():
        if isinstance(m, nn.Conv2d):
            yield m

def make_structurally_pruned_copy(src_model: nn.Module, total_fraction: float) -> nn.Module:
    
    # deepcopy -> prune filters -> remove reparam -> return
    m = copy.deepcopy(src_model).eval()
    for conv in conv_layers_only(m):
        prune.ln_structured(conv, name="weight", amount=total_fraction, n=1, dim=0)  # prune filters
        prune.remove(conv, "weight")  # make permanent
    return m

def run_structured_once(pct: float):
    pruned = make_structurally_pruned_copy(model, pct)
    print(f"Pruned {int(pct*100)}% of convolution filters:")
    test_model(pruned, test_data, mode="torch")
    print()

for p in PCTS:
    run_structured_once(p)

# *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
################################################################################
#                              END OF YOUR CODE                                #
################################################################################

## Inline Question 2:

What is the difference between performing Structured Pruning vs Dropout ? 
Why would it be beneficial to perform both techniques when developing a Neural Network?


## Answer: 

[FILL YOU ANSWER HERE]
